# Session 4: AI in Cybersecurity Policies

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hamnakhan123/Cybersecurity-and-Digital-Forensics/blob/main/term-1/05-cyber-security-frameworks/code/session-04-ai-in-cybersecurity-policies.ipynb)

*Cybersecurity and Digital Forensics* · Prepared by **Hamna Zahid Khan**, Junior Lecturer

Run each cell from top to bottom (Shift+Enter). Every example is self-contained and safe to run in a lab or on Colab.

## `session-04_login_anomaly.py`

In [1]:
"""Session 4 - User Behaviour Analytics: spotting unusual logins and enforcing policy.

An AI-driven security tool does not start with a list of "bad" events. It first LEARNS
what is normal for each user (a baseline), then scores how far a new event is from that
normal, and finally applies the organisation's POLICY to the score. This script shows
the three steps with plain statistics, so every alert can be explained:

  1. Baseline  - per user: usual login hours (mean and spread), cities, devices, data volume.
  2. Score     - add points for each unusual feature; keep the reason for each point.
  3. Enforce   - the policy turns the score into ALLOW, STEP-UP MFA or BLOCK + ALERT.

All users, cities and devices are fictional (Clifton Institute of Technology, Karachi).
Run:  python session-04_login_anomaly.py
"""
import random
from statistics import mean, pstdev

rng = random.Random(42)  # fixed seed: the same "history" every run

# ---------- 1. Fictional 30-day login history -> baseline ----------
PROFILES = {  # user: (usual hour, hour spread, cities, devices, usual MB downloaded)
    "sana.faculty":  (9, 1.0, ["Karachi"], ["LAPTOP-SANA"], 40),
    "omer.registrar": (10, 1.5, ["Karachi"], ["PC-REG-02"], 25),
    "bilal.student": (21, 2.0, ["Karachi", "Hyderabad"], ["PHONE-BILAL", "LAB-WS-14"], 60),
}

history = []
for user, (hour, spread, cities, devices, mb) in PROFILES.items():
    for day in range(30):
        h = max(0, min(23, round(rng.gauss(hour, spread))))
        history.append({"user": user, "hour": h, "city": rng.choice(cities),
                        "device": rng.choice(devices), "mb": max(1, round(rng.gauss(mb, mb * 0.25)))})


def baseline(user):
    rows = [r for r in history if r["user"] == user]
    hours, mbs = [r["hour"] for r in rows], [r["mb"] for r in rows]
    return {"h_mean": mean(hours), "h_sd": max(pstdev(hours), 0.5),
            "mb_mean": mean(mbs), "mb_sd": max(pstdev(mbs), 1.0),
            "cities": {r["city"] for r in rows}, "devices": {r["device"] for r in rows}}


BASE = {u: baseline(u) for u in PROFILES}

# ---------- 2. Score new events (explainable points) ----------
POLICY = {"step_up_at": 30, "block_at": 60}   # thresholds set by the security policy


def score(event):
    b, pts, why = BASE[event["user"]], 0, []
    z_hour = abs(event["hour"] - b["h_mean"]) / b["h_sd"]
    if z_hour > 3:
        pts += 25; why.append(f"login hour {event['hour']:02d}:00 is {z_hour:.1f} SD from usual")
    if event["city"] not in b["cities"]:
        pts += 30; why.append(f"new location {event['city']}")
    if event["device"] not in b["devices"]:
        pts += 20; why.append(f"unknown device {event['device']}")
    z_mb = (event["mb"] - b["mb_mean"]) / b["mb_sd"]
    if z_mb > 3:
        pts += 25; why.append(f"downloaded {event['mb']} MB ({z_mb:.1f} SD above usual)")
    if event.get("failed_before", 0) >= 5:
        pts += 15; why.append(f"{event['failed_before']} failed attempts just before")
    return min(pts, 100), why


def enforce(pts):
    if pts >= POLICY["block_at"]:
        return "BLOCK + ALERT SOC"
    if pts >= POLICY["step_up_at"]:
        return "STEP-UP MFA"
    return "ALLOW"


NEW_EVENTS = [
    {"id": "E1", "user": "sana.faculty", "hour": 9, "city": "Karachi", "device": "LAPTOP-SANA", "mb": 45},
    {"id": "E2", "user": "omer.registrar", "hour": 23, "city": "Karachi", "device": "PC-REG-02", "mb": 30},
    {"id": "E3", "user": "bilal.student", "hour": 22, "city": "Lahore", "device": "PHONE-BILAL", "mb": 70},
    {"id": "E4", "user": "omer.registrar", "hour": 3, "city": "Frankfurt", "device": "UNKNOWN-VM",
     "mb": 900, "failed_before": 7},
    {"id": "E5", "user": "sana.faculty", "hour": 14, "city": "Karachi", "device": "TABLET-NEW", "mb": 38},
]

print("[1] Learned baselines (30 days of fictional history)")
for u, b in BASE.items():
    print(f"  {u:<15} usual hour {b['h_mean']:4.1f} +/- {b['h_sd']:.1f} | "
          f"~{b['mb_mean']:.0f} MB | cities {sorted(b['cities'])}")

print(f"\n[2] Scoring new logins (policy: step-up at {POLICY['step_up_at']}, block at {POLICY['block_at']})")
counts = {}
for e in NEW_EVENTS:
    pts, why = score(e)
    action = enforce(pts)
    counts[action] = counts.get(action, 0) + 1
    print(f"  {e['id']} {e['user']:<15} score {pts:3d} -> {action}")
    for w in why:
        print(f"       - {w}")

print("\n[3] Decision summary:", ", ".join(f"{k}: {v}" for k, v in counts.items()))
print("Note: a human analyst reviews every BLOCK; the score only recommends.")

[1] Learned baselines (30 days of fictional history)
  sana.faculty    usual hour  9.2 +/- 1.0 | ~39 MB | cities ['Karachi']
  omer.registrar  usual hour 10.0 +/- 1.3 | ~23 MB | cities ['Karachi']
  bilal.student   usual hour 21.0 +/- 1.8 | ~59 MB | cities ['Hyderabad', 'Karachi']

[2] Scoring new logins (policy: step-up at 30, block at 60)
  E1 sana.faculty    score   0 -> ALLOW
  E2 omer.registrar  score  25 -> ALLOW
       - login hour 23:00 is 9.7 SD from usual
  E3 bilal.student   score  30 -> STEP-UP MFA
       - new location Lahore
  E4 omer.registrar  score 100 -> BLOCK + ALERT SOC
       - login hour 03:00 is 5.2 SD from usual
       - new location Frankfurt
       - unknown device UNKNOWN-VM
       - downloaded 900 MB (153.6 SD above usual)
       - 7 failed attempts just before
  E5 sana.faculty    score  45 -> STEP-UP MFA
       - login hour 14:00 is 4.8 SD from usual
       - unknown device TABLET-NEW

[3] Decision summary: ALLOW: 2, STEP-UP MFA: 2, BLOCK + ALERT SOC: 1
No

## `session-04_phishing_classifier.py`

In [2]:
"""Session 4 - A tiny, explainable phishing classifier (Naive Bayes from scratch).

Email security gateways use machine learning to decide whether a message is phishing.
This script trains a very small Naive Bayes model on 16 fictional, harmless emails,
tests it on 6 unseen ones, and then applies an EMAIL POLICY:

  probability >= 0.90  -> QUARANTINE (user never sees it, admin can release)
  0.50 to 0.90         -> DELIVER WITH WARNING BANNER
  below 0.50           -> DELIVER

It also prints the words that pushed each decision, because a policy that cannot be
explained cannot be audited. No real brands, links or payloads are used.
Run:  python session-04_phishing_classifier.py
"""
import math
import re
from collections import Counter

TRAIN = [  # (label, text)  1 = phishing, 0 = legitimate
    (1, "urgent verify your account password now or it will be suspended"),
    (1, "your mailbox is full click here to verify login immediately"),
    (1, "congratulations you won a prize claim your reward by entering bank details"),
    (1, "security alert unusual sign in confirm your password at this link"),
    (1, "invoice overdue pay now using the attached link to avoid penalty"),
    (1, "hr update confirm your salary account details today urgent"),
    (1, "your parcel is held pay a small fee click the link now"),
    (1, "final warning account locked verify identity within 24 hours"),
    (0, "reminder the data structures lab moves to room 204 on monday"),
    (0, "please find the agenda for the faculty meeting on thursday"),
    (0, "the library will close early on friday for maintenance"),
    (0, "assignment two marks are uploaded to the course portal"),
    (0, "thank you for attending the workshop slides are attached"),
    (0, "the exam timetable for the midterm week has been published"),
    (0, "lunch for the department seminar will be served at one"),
    (0, "please review the draft syllabus and send comments by friday"),
]
TEST = [
    (1, "urgent your password expires today click here to verify"),
    (1, "claim your bonus reward enter your bank account details"),
    (0, "the programming lab quiz is on wednesday in room 204"),
    (0, "seminar slides and agenda are attached for review"),
    (1, "account suspended confirm login details at the link"),
    (0, "please verify the attendance sheet before friday"),  # tricky: has "verify"
]
STOP = {"the", "a", "to", "your", "is", "on", "for", "at", "by", "or", "it", "in", "and", "of", "be", "will", "has", "been", "are", "this", "using", "now"}


def words(text):
    return [w for w in re.findall(r"[a-z0-9]+", text.lower()) if w not in STOP]


class NaiveBayes:
    def fit(self, data):
        self.counts = {0: Counter(), 1: Counter()}
        self.docs = Counter(label for label, _ in data)
        for label, text in data:
            self.counts[label].update(words(text))
        self.vocab = set(self.counts[0]) | set(self.counts[1])
        self.total = {c: sum(self.counts[c].values()) for c in (0, 1)}
        return self

    def word_logratio(self, w):  # > 0 means the word points to phishing
        p = [(self.counts[c][w] + 1) / (self.total[c] + len(self.vocab)) for c in (0, 1)]
        return math.log(p[1] / p[0])

    def predict_proba(self, text):
        log_odds = math.log(self.docs[1] / self.docs[0])
        log_odds += sum(self.word_logratio(w) for w in words(text) if w in self.vocab)
        return 1 / (1 + math.exp(-log_odds))


def policy(p):
    return "QUARANTINE" if p >= 0.90 else "WARN BANNER" if p >= 0.50 else "DELIVER"


model = NaiveBayes().fit(TRAIN)
print(f"[1] Trained on {len(TRAIN)} emails, vocabulary {len(model.vocab)} words")
top = sorted(model.vocab, key=lambda w: (-model.word_logratio(w), w))[:6]
print("    strongest phishing words:", ", ".join(top))

print("\n[2] Test emails -> probability -> policy action")
tp = fp = tn = fn = 0
for label, text in TEST:
    p = model.predict_proba(text)
    pred = 1 if p >= 0.50 else 0
    tp += pred == 1 and label == 1; fp += pred == 1 and label == 0
    tn += pred == 0 and label == 0; fn += pred == 0 and label == 1
    clues = sorted({w for w in words(text) if w in model.vocab},
                   key=lambda w: (-model.word_logratio(w), w))[:2]
    mark = "ok " if pred == label else "ERR"
    print(f"  {mark} p={p:.2f} {policy(p):<11} | {text[:46]:<46} | clues: {', '.join(clues)}")

acc = (tp + tn) / len(TEST)
prec = tp / (tp + fp) if tp + fp else 0
rec = tp / (tp + fn) if tp + fn else 0
print(f"\n[3] Confusion: TP={tp} FP={fp} TN={tn} FN={fn}")
print(f"    accuracy {acc:.2f} | precision {prec:.2f} | recall {rec:.2f}")
print("    A false positive delays real mail; a false negative lets phishing in.")
print("    Policy owners choose thresholds knowing which mistake costs more.")

[1] Trained on 16 emails, vocabulary 91 words
    strongest phishing words: account, link, verify, click, confirm, details

[2] Test emails -> probability -> policy action
  ok  p=1.00 QUARANTINE  | urgent your password expires today click here  | clues: verify, click
  ok  p=0.98 QUARANTINE  | claim your bonus reward enter your bank accoun | clues: account, details
  ok  p=0.09 DELIVER     | the programming lab quiz is on wednesday in ro | clues: 204, lab
  ok  p=0.04 DELIVER     | seminar slides and agenda are attached for rev | clues: attached, agenda
  ok  p=1.00 QUARANTINE  | account suspended confirm login details at the | clues: account, link
  ok  p=0.26 DELIVER     | please verify the attendance sheet before frid | clues: verify, friday

[3] Confusion: TP=3 FP=0 TN=3 FN=0
    accuracy 1.00 | precision 1.00 | recall 1.00
    A false positive delays real mail; a false negative lets phishing in.
    Policy owners choose thresholds knowing which mistake costs more.


## `session-04_adaptive_access.py`

In [3]:
"""Session 4 - Adaptive (risk-based) access control written as policy-as-code.

Traditional access control asks one question: "Is the password right?" Adaptive access,
used in Zero Trust and modern IAM products, asks: "Given everything we know right now,
how risky is this request, and how sensitive is what is being requested?"

The script combines:
  - signals: device compliance, network, location, time, and a behaviour (UBA) risk score
  - resource sensitivity: public, internal, confidential, restricted
  - rules from a written policy (each rule has an ID so auditors can trace decisions)
and returns ALLOW, ALLOW WITH MFA, ALLOW READ-ONLY or DENY, with an audit line per request.
All people and systems are fictional.
Run:  python session-04_adaptive_access.py
"""

SENSITIVITY = {"public": 0, "internal": 1, "confidential": 2, "restricted": 3}
RESOURCES = {"Course portal": "internal", "Grades database": "restricted",
             "Library catalogue": "public", "HR payroll": "confidential"}


def risk(sig):
    """Return (risk 0-100, reasons). Weights come from the written access policy."""
    r, why = 0, []
    if not sig["device_compliant"]:
        r += 30; why.append("device not compliant (no disk encryption or old patches)")
    if sig["network"] == "public_wifi":
        r += 15; why.append("public Wi-Fi")
    if sig["country"] != "PK":
        r += 20; why.append(f"outside usual country ({sig['country']})")
    if not 7 <= sig["hour"] <= 22:
        r += 10; why.append(f"outside working hours ({sig['hour']:02d}:00 PKT)")
    r += sig["uba_score"] // 2
    if sig["uba_score"] >= 40:
        why.append(f"behaviour analytics score {sig['uba_score']}")
    return min(r, 100), why


def decide(sig, resource):
    level = SENSITIVITY[RESOURCES[resource]]
    r, why = risk(sig)
    # Rules are checked in order; the first match wins (like a firewall).
    if sig["role"] not in sig["allowed_roles"]:
        return "DENY", "AC-1 role not authorised for resource", r, why
    if r >= 70:
        return "DENY", "AC-2 risk too high for any access", r, why
    if level == 3 and (r >= 30 or not sig["device_compliant"]):
        return "DENY", "AC-3 restricted data needs compliant device and low risk", r, why
    if level >= 2 and r >= 40:
        return "ALLOW READ-ONLY", "AC-4 medium risk: no edits or downloads", r, why
    if level >= 1 and (r >= 20 or level >= 2):
        return "ALLOW WITH MFA", "AC-5 step-up authentication", r, why
    return "ALLOW", "AC-6 public data or low risk", r, why


BASE = {"device_compliant": True, "network": "campus", "country": "PK", "hour": 10, "uba_score": 5}
REQUESTS = [
    ("REQ-01", "Ms. Sana (faculty)", "Grades database", dict(BASE, role="faculty", allowed_roles={"faculty", "registrar"})),
    ("REQ-02", "Ms. Sana (faculty)", "Grades database", dict(BASE, role="faculty", allowed_roles={"faculty", "registrar"},
                                                             network="public_wifi", hour=23)),
    ("REQ-03", "Bilal (student)", "Course portal", dict(BASE, role="student", allowed_roles={"student", "faculty"},
                                                        network="public_wifi")),
    ("REQ-04", "Bilal (student)", "Grades database", dict(BASE, role="student", allowed_roles={"faculty", "registrar"})),
    ("REQ-05", "Mr. Omer (HR)", "HR payroll", dict(BASE, role="hr", allowed_roles={"hr"}, country="AE", uba_score=50)),
    ("REQ-06", "Mr. Omer (HR)", "HR payroll", dict(BASE, role="hr", allowed_roles={"hr"}, device_compliant=False,
                                                   country="DE", hour=3, uba_score=80)),
    ("REQ-07", "Visitor", "Library catalogue", dict(BASE, role="guest", allowed_roles={"guest", "student", "faculty"},
                                                   device_compliant=False, network="public_wifi")),
]

print(f"{'Req':<7}{'User':<20}{'Resource':<18}{'Risk':>5}  Decision")
audit = []
for rid, user, res, sig in REQUESTS:
    decision, rule, r, why = decide(sig, res)
    print(f"{rid:<7}{user:<20}{res:<18}{r:>5}  {decision}")
    audit.append(f"{rid} | {rule} | factors: {'; '.join(why) if why else 'none'}")

print("\nAudit log (kept for compliance review):")
for line in audit:
    print("  " + line)
print("\nHuman oversight: users who are denied can raise a ticket; an analyst can override,")
print("and every override is logged with the analyst's name and reason.")

Req    User                Resource           Risk  Decision
REQ-01 Ms. Sana (faculty)  Grades database       2  ALLOW WITH MFA
REQ-02 Ms. Sana (faculty)  Grades database      27  ALLOW WITH MFA
REQ-03 Bilal (student)     Course portal        17  ALLOW
REQ-04 Bilal (student)     Grades database       2  DENY
REQ-05 Mr. Omer (HR)       HR payroll           45  ALLOW READ-ONLY
REQ-06 Mr. Omer (HR)       HR payroll          100  DENY
REQ-07 Visitor             Library catalogue    47  ALLOW

Audit log (kept for compliance review):
  REQ-01 | AC-5 step-up authentication | factors: none
  REQ-02 | AC-5 step-up authentication | factors: public Wi-Fi; outside working hours (23:00 PKT)
  REQ-03 | AC-6 public data or low risk | factors: public Wi-Fi
  REQ-04 | AC-1 role not authorised for resource | factors: none
  REQ-05 | AC-4 medium risk: no edits or downloads | factors: outside usual country (AE); behaviour analytics score 50
  REQ-06 | AC-2 risk too high for any access | factors: device no

## `session-04_ai_governance_metrics.py`

In [4]:
"""Session 4 - Governing an AI security model: accuracy, fairness and drift.

A security policy that relies on AI must say how the AI itself is checked. This script
plays the role of the monthly "model review" for a fictional login-anomaly model at
Clifton Institute of Technology (Karachi) and reports against written thresholds:

  1. Accuracy  - accuracy, precision, recall and false-positive rate (FPR)
  2. Fairness  - do two user groups (staff, students) get the same treatment?
                 FPR gap and flag-rate ratio (the "four-fifths" or 80% rule of thumb)
  3. Drift     - has the data changed since training? Population Stability Index (PSI)
                 plus the weekly accuracy trend
  4. Scorecard - PASS / FAIL against the AI governance policy, with actions

All numbers are fictional and chosen for teaching.
Run:  python session-04_ai_governance_metrics.py
"""
import math

# ---------- Fictional evaluation set: 200 labelled login events ----------
# Built from confusion counts so the result is fully reproducible.
COUNTS = {  # group: (TP, FN, FP, TN)   positive = "malicious login"
    "staff":   (18, 2, 3, 77),
    "student": (17, 3, 12, 68),
}
records = []  # (group, actual, predicted)
for g, (tp, fn, fp, tn) in COUNTS.items():
    records += [(g, 1, 1)] * tp + [(g, 1, 0)] * fn + [(g, 0, 1)] * fp + [(g, 0, 0)] * tn


def metrics(rows):
    tp = sum(1 for _, a, p in rows if a == 1 and p == 1)
    fn = sum(1 for _, a, p in rows if a == 1 and p == 0)
    fp = sum(1 for _, a, p in rows if a == 0 and p == 1)
    tn = sum(1 for _, a, p in rows if a == 0 and p == 0)
    n = len(rows)
    return {"n": n, "accuracy": (tp + tn) / n, "precision": tp / (tp + fp),
            "recall": tp / (tp + fn), "fpr": fp / (fp + tn), "flag_rate": (tp + fp) / n}


overall = metrics(records)
by_group = {g: metrics([r for r in records if r[0] == g]) for g in COUNTS}

print("[1] Accuracy on 200 labelled events")
print(f"  {'group':<8}{'n':>4}{'acc':>7}{'prec':>7}{'recall':>8}{'FPR':>7}{'flagged':>9}")
for name, m in [("ALL", overall)] + list(by_group.items()):
    print(f"  {name:<8}{m['n']:>4}{m['accuracy']:>7.2f}{m['precision']:>7.2f}"
          f"{m['recall']:>8.2f}{m['fpr']:>7.2f}{m['flag_rate']:>9.2f}")

# ---------- 2. Fairness ----------
fpr_gap = abs(by_group["student"]["fpr"] - by_group["staff"]["fpr"])
rates = [by_group[g]["flag_rate"] for g in COUNTS]
flag_ratio = min(rates) / max(rates)
print("\n[2] Fairness")
print(f"  FPR gap (student - staff): {fpr_gap:.3f}  -> innocent students are flagged "
      f"{by_group['student']['fpr'] / by_group['staff']['fpr']:.1f}x as often")
print(f"  Flag-rate ratio: {flag_ratio:.2f}  (below 0.80 suggests unequal treatment)")

# ---------- 3. Drift ----------
BINS = ["00-06", "06-12", "12-18", "18-24"]
train_share = [0.05, 0.45, 0.35, 0.15]   # login hours when the model was trained
now_share = [0.08, 0.30, 0.32, 0.30]     # this month: evening classes added


def psi(expected, actual):
    return sum((a - e) * math.log(a / e) for e, a in zip(expected, actual))


drift = psi(train_share, now_share)
print("\n[3] Drift in login-hour distribution")
for b, e, a in zip(BINS, train_share, now_share):
    print(f"  {b}  train {e:4.0%}  now {a:4.0%}  {'#' * round(a * 40)}")
print(f"  PSI = {drift:.3f}  (<0.10 stable, 0.10-0.25 monitor, >0.25 retrain)")

WEEKLY = [(186, 200), (184, 200), (182, 200), (176, 200), (170, 200), (164, 200)]
trend = [c / t for c, t in WEEKLY]
print("  Weekly accuracy:", " ".join(f"W{i + 1}={a:.2f}" for i, a in enumerate(trend)))

# ---------- 4. Scorecard against the governance policy ----------
POLICY = [  # (check, value, passes?, action if it fails)
    ("Accuracy >= 0.85", overall["accuracy"], overall["accuracy"] >= 0.85, "retrain / tune"),
    ("Recall >= 0.80", overall["recall"], overall["recall"] >= 0.80, "lower alert threshold"),
    ("Precision >= 0.70", overall["precision"], overall["precision"] >= 0.70, "reduce false alarms"),
    ("FPR gap <= 0.05", fpr_gap, fpr_gap <= 0.05, "add group-aware baselines, review features"),
    ("Flag-rate ratio >= 0.80", flag_ratio, flag_ratio >= 0.80, "bias review before next release"),
    ("PSI < 0.10", drift, drift < 0.10, "monitor weekly; retrain if PSI > 0.25"),
    ("Latest weekly accuracy >= 0.85", trend[-1], trend[-1] >= 0.85, "retrain on recent data"),
]
print("\n[4] Governance scorecard")
fails = 0
for check, value, ok, action in POLICY:
    fails += not ok
    print(f"  {'PASS' if ok else 'FAIL'}  {check:<31} value {value:.3f}" + ("" if ok else f"  -> {action}"))
verdict = "APPROVED" if fails == 0 else "APPROVED WITH CONDITIONS" if fails <= 2 else "NOT APPROVED"
print(f"\n  {fails} of {len(POLICY)} checks failed -> model status: {verdict}")
print("  Decision owner: CISO with the AI governance committee (human sign-off).")

[1] Accuracy on 200 labelled events
  group      n    acc   prec  recall    FPR  flagged
  ALL      200   0.90   0.70    0.88   0.09     0.25
  staff    100   0.95   0.86    0.90   0.04     0.21
  student  100   0.85   0.59    0.85   0.15     0.29

[2] Fairness
  FPR gap (student - staff): 0.112  -> innocent students are flagged 4.0x as often
  Flag-rate ratio: 0.72  (below 0.80 suggests unequal treatment)

[3] Drift in login-hour distribution
  00-06  train   5%  now   8%  ###
  06-12  train  45%  now  30%  ############
  12-18  train  35%  now  32%  #############
  18-24  train  15%  now  30%  ############
  PSI = 0.182  (<0.10 stable, 0.10-0.25 monitor, >0.25 retrain)
  Weekly accuracy: W1=0.93 W2=0.92 W3=0.91 W4=0.88 W5=0.85 W6=0.82

[4] Governance scorecard
  PASS  Accuracy >= 0.85                value 0.900
  PASS  Recall >= 0.80                  value 0.875
  PASS  Precision >= 0.70               value 0.700
  FAIL  FPR gap <= 0.05                 value 0.112  -> add group-aware

## `mini_project_ai_compliance_toolkit.py`

In [5]:
"""Module 5 mini project - AI security compliance toolkit.

Before an organisation lets AI enforce its security policies, it should keep a
REGISTER of every AI system and check each one against a set of controls. This toolkit:

  1. assigns a risk tier to each system (simplified from the EU AI Act idea of
     minimal / limited / high risk; education and employment uses are high risk),
  2. scores each system against 11 controls mapped to the NIST AI RMF functions
     (GOVERN, MAP, MEASURE, MANAGE) and to GDPR / EU AI Act / ISO/IEC 42001 ideas,
  3. prints a compliance percentage, a RAG rating and a prioritised action plan.

Status codes: Y = done, P = partly done, N = not done, - = not applicable.
High-risk systems must have every weight-3 control at Y to be "READY".
Teaching simplification, not legal advice. All systems are fictional.
Run:  python mini_project_ai_compliance_toolkit.py
"""

CONTROLS = [  # (id, control, NIST AI RMF function, reference, weight 1-3)
    ("C01", "Owner, purpose and scope documented", "GOVERN", "ISO/IEC 42001 roles and AI policy", 2),
    ("C02", "Impact assessment / DPIA completed", "MAP", "GDPR Art 35; AI Act risk management", 3),
    ("C03", "Training data checked for quality and bias", "MAP", "AI Act Art 10 data governance", 3),
    ("C04", "Accuracy and fairness tested before release", "MEASURE", "AI Act Art 15; NIST MEASURE", 3),
    ("C05", "Drift monitored in production", "MEASURE", "NIST MEASURE; ISO/IEC 42001 monitoring", 2),
    ("C06", "Human review of high-impact decisions", "MANAGE", "GDPR Art 22; AI Act Art 14", 3),
    ("C07", "Decisions explainable and logged", "MANAGE", "AI Act Art 12; GDPR accountability", 2),
    ("C08", "Adversarial / prompt-injection testing", "MEASURE", "AI Act Art 15; MITRE ATLAS; OWASP LLM Top 10", 2),
    ("C09", "People told when AI is used", "GOVERN", "AI Act Art 50; GDPR Art 13-14", 1),
    ("C10", "Data minimised, retention period set", "MAP", "GDPR Art 5", 2),
    ("C11", "AI incident response and rollback plan", "MANAGE", "ISO/IEC 42001; NIST MANAGE", 2),
]

SYSTEMS = {  # name: (use, interacts with public?, decides about people in education/jobs?, statuses C01..C11)
    "UBA login model": ("flags risky logins, forces MFA", False, False,
                        "Y Y P P P Y Y N P Y P"),
    "Email phishing filter": ("quarantines suspicious email", False, False,
                              "Y P Y Y Y P Y P - Y Y"),
    "Exam proctoring AI": ("flags possible cheating in online exams", True, True,
                           "Y N N P N P P N Y P N"),
    "IT helpdesk chatbot": ("answers password and VPN questions", True, False,
                            "Y Y P Y P - P P N Y Y"),
}
VALUE = {"Y": 1.0, "P": 0.5, "N": 0.0}


def tier(public, high_impact):
    if high_impact:
        return "HIGH"
    return "LIMITED (transparency duties)" if public else "MINIMAL"


def assess(statuses, risk_tier):
    got = possible = 0
    gaps = []
    for (cid, name, fn, ref, w), s in zip(CONTROLS, statuses.split()):
        if s == "-":
            continue
        possible += w
        got += w * VALUE[s]
        if s != "Y":
            priority = w + (2 if risk_tier == "HIGH" else 0) + (1 if s == "N" else 0)
            gaps.append((priority, cid, name, fn, ref, s))
    pct = 100 * got / possible
    weight3 = {c[0] for c in CONTROLS if c[4] == 3}
    blocked = risk_tier == "HIGH" and any(g[1] in weight3 for g in gaps)
    rag = "GREEN" if pct >= 85 else "AMBER" if pct >= 60 else "RED"
    return pct, rag, blocked, sorted(gaps, reverse=True)


print("AI SYSTEM REGISTER - Clifton Institute of Technology (fictional)\n")
print(f"{'System':<23}{'Tier':<31}{'Score':>6}  Rating")
results = {}
for name, (use, public, high, statuses) in SYSTEMS.items():
    t = tier(public, high)
    pct, rag, blocked, gaps = assess(statuses, t)
    results[name] = (t, gaps, blocked)
    status = "NOT READY" if blocked else rag
    print(f"{name:<23}{t:<31}{pct:5.0f}%  {status}")

print("\nCoverage by NIST AI RMF function (share of controls fully done, all systems):")
for fn in ("GOVERN", "MAP", "MEASURE", "MANAGE"):
    idx = [i for i, c in enumerate(CONTROLS) if c[2] == fn]
    cells = [SYSTEMS[s][3].split()[i] for s in SYSTEMS for i in idx]
    applicable = [c for c in cells if c != "-"]
    done = sum(c == "Y" for c in applicable)
    print(f"  {fn:<8} {done:>2}/{len(applicable):<2} {'#' * done}")

print("\nTop actions (highest priority first):")
plan = [(p, name, cid, ctl, ref, s) for name, (t, gaps, _) in results.items()
        for p, cid, ctl, fn, ref, s in gaps]
for p, name, cid, ctl, ref, s in sorted(plan, reverse=True)[:8]:
    print(f"  P{p} {name:<22} {cid} {ctl} [{s}] ({ref})")

print("\nRule: a HIGH-risk system marked NOT READY must not make decisions on its own;")
print("its flags go to a trained human reviewer until every weight-3 control is done.")

AI SYSTEM REGISTER - Clifton Institute of Technology (fictional)

System                 Tier                            Score  Rating
UBA login model        MINIMAL                           70%  AMBER
Email phishing filter  MINIMAL                           83%  AMBER
Exam proctoring AI     HIGH                              32%  NOT READY
IT helpdesk chatbot    LIMITED (transparency duties)     75%  AMBER

Coverage by NIST AI RMF function (share of controls fully done, all systems):
  GOVERN    5/7  #####
  MAP       6/12 ######
  MEASURE   3/12 ###
  MANAGE    5/11 #####

Top actions (highest priority first):
  P6 Exam proctoring AI     C03 Training data checked for quality and bias [N] (AI Act Art 10 data governance)
  P6 Exam proctoring AI     C02 Impact assessment / DPIA completed [N] (GDPR Art 35; AI Act risk management)
  P5 Exam proctoring AI     C11 AI incident response and rollback plan [N] (ISO/IEC 42001; NIST MANAGE)
  P5 Exam proctoring AI     C08 Adversarial / prompt-inj